In [1]:
import os
import sys
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
import lightgbm as lgbm

from hyperopt import hp
from hyperopt import fmin, tpe, Trials, STATUS_OK

from utils import user_utils
from utils import preprocessing

In [2]:
# data loading
train_df ,test_df= preprocessing.load_data()
test_df = test_df.drop(["ID"], axis=1)

In [3]:
X_features, y_target = preprocessing.split_features_target(train_df)

In [4]:
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [5]:
# 컬럼 삭제 ,결측치 ,log1p 처리 및 스탠다드 스케일처
# from common.preprocessing import load_data
scaled_X_train, scaled_X_test, scaler = preprocessing.scale_data(X_features, test_df)

In [6]:
X_train, X_test, y_train, y_test = preprocessing.data_split(scaled_X_train, y_target)

In [ ]:
# --- 2. Hyperparameter 탐색 공간 정의 ---

# XGBoost 탐색 공간
xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 5, 15, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 10),
    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5)
}

# LightGBM 탐색 공간
lgbm_search_space = {
    'num_leaves': hp.quniform('num_leaves', 32, 128, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 10),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1),
}

# RandomForest 탐색 공간
rf_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 10),
    'max_depth': hp.quniform('max_depth', 10, 30, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 8, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 12, 1)
}
